# 📖 How2Sign Continuous Training Pipeline

**Purpose:** Train a Continuous Sign Language Recognition (CSLR) model using the How2Sign dataset.
This pipeline loads pre-extracted OpenPose keypoints from `.json` files and trains a Seq2Seq BiLSTM to output sequences of words.

In [6]:
import math
import h5py
# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
import os
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from pathlib import Path

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f'GPU mode: {len(gpus)} GPU(s), mixed_float16')
else:
    print('CPU mode')



GPU mode: 1 GPU(s), mixed_float16


## 2. Dataset Paths
Pointing to the downloaded CSV and JSON artifacts.

In [7]:
# ============================================================
# 2. DATASET PATHS  (Kaggle – How2Sign-keypoints dataset)
# ============================================================
BASE_DIR = Path('/kaggle/input/datasets/nazarboholii/how2sign')

CSV_TRAIN = BASE_DIR / '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'
CSV_VAL   = BASE_DIR / '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv'
CSV_TEST  = BASE_DIR / '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv'

# JSON frames live one folder deeper than the split root
JSON_DIR_TRAIN = BASE_DIR / 'train_2D_keypoints' / 'openpose_output' / 'json'
JSON_DIR_VAL   = BASE_DIR / 'val_2D_keypoints'   / 'openpose_output' / 'json'
JSON_DIR_TEST  = BASE_DIR / 'test_2D_keypoints'  / 'openpose_output' / 'json'

for label, p in [('BASE_DIR', BASE_DIR),
                 ('CSV_TRAIN', CSV_TRAIN), ('CSV_VAL', CSV_VAL), ('CSV_TEST', CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")


BASE_DIR     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign
CSV_TRAIN    exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv
CSV_VAL      exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv
CSV_TEST     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv
JSON_TRAIN   exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/train_2D_keypoints/openpose_output/json
JSON_VAL     exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/val_2D_keypoints/openpose_output/json
JSON_TEST    exists=False  → /kaggle/input/datasets/nazarboholii/how2sign/test_2D_keypoints/openpose_output/json


## 3. Custom Data Generator
This generator reads the OpenPose JSON files batch-by-batch to prevent out-of-memory errors.

In [8]:
# ============================================================
# 3. CUSTOM DATA GENERATOR (OpenPose JSON → 78-dim feature vector)
# ============================================================
from tensorflow.keras.utils import Sequence

# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------
NUM_BODY_KPS  = 25   # OpenPose body_25 model
NUM_HAND_KPS  = 21   # each hand
# 78-dim = body (x,y) × 25 + left-hand wrist (x,y) × 4
#        = 50 + 28  — we use (x,y) of 14 selected hand keypoints
# Simpler split actually used here:
#   pose_keypoints_2d  → 25 × 3 = 75 values  (x, y, conf per joint)
#   face center (nose) → 3 more values        → total 78
NOSE_IDX = 0   # index in pose_keypoints_2d for nose keypoint (body_25)

def convert_openpose_to_78dim(frame_data: dict) -> np.ndarray:
    """
    Parse one OpenPose frame JSON dict and return a (78,) float32 array.

    OpenPose frame JSON structure:
        { "version": ...,
          "people": [
            { "pose_keypoints_2d":  [x0,y0,c0, x1,y1,c1, ...],  # 25 kps × 3 = 75
              "face_keypoints_2d":  [...],
              "hand_left_keypoints_2d":  [...],
              "hand_right_keypoints_2d": [...] }
          ]
        }
    We extract the 75 body values + the 3 face-nose values (or zeros) = 78.
    """
    feat = np.zeros(78, dtype=np.float32)
    people = frame_data.get('people', [])
    if not people:
        return feat

    person = people[0]

    # --- body pose: 25 kps × 3 = 75 values ---
    pose_raw = person.get('pose_keypoints_2d', [])
    pose_arr = np.array(pose_raw, dtype=np.float32)
    n_body = min(len(pose_arr), 75)
    feat[:n_body] = pose_arr[:n_body]

    # --- face: use the nose keypoint (index 0 in face_keypoints_2d) ---
    face_raw = person.get('face_keypoints_2d', [])
    if len(face_raw) >= 3:
        feat[75:78] = np.array(face_raw[:3], dtype=np.float32)

    return feat


def load_sentence_frames(sentence_name: str, json_dir: Path) -> list:
    """
    Return a list of (78,) arrays – one per frame – for the given sentence.
    Frame JSON files are stored at:
        <json_dir>/<sentence_name>/<sentence_name>_XXXXXX_keypoints.json
    """
    sentence_dir = json_dir / sentence_name
    if not sentence_dir.exists():
        return []

    frame_files = sorted(sentence_dir.glob('*.json'))
    frames = []
    for fp in frame_files:
        with fp.open('r') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
        frames.append(convert_openpose_to_78dim(data))
    return frames


# ---------------------------------------------------------------------------
# Generator
# ---------------------------------------------------------------------------
class How2SignGenerator(Sequence):
    """Keras Sequence generator for How2Sign continuous sign recognition."""

    # Expected CSV columns (How2Sign realigned format)
    COL_NAME     = 'SENTENCE_NAME'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path: Path, json_dir: Path,
                 tokenizer=None,
                 batch_size: int = 8,
                 sequence_length: int = 150,
                 num_features: int = 78):

        if not csv_path.exists():
            print(f'⚠️  CSV not found: {csv_path}')
            self.df = pd.DataFrame()
        else:
            self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            # Fallback: try comma-separated
            if self.COL_NAME not in self.df.columns:
                self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            print(f'✅ Loaded {len(self.df):,} rows from {csv_path.name}')
            print(f'   Columns: {list(self.df.columns)}')

        self.json_dir        = json_dir
        self.tokenizer       = tokenizer
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.num_features    = num_features

    # ---- helpers -----------------------------------------------------------

    def _pad_frames(self, frames: list) -> np.ndarray:
        """Pad / truncate frame list to (sequence_length, num_features)."""
        out = np.zeros((self.sequence_length, self.num_features), dtype=np.float32)
        T = min(len(frames), self.sequence_length)
        if T > 0:
            out[:T] = np.stack(frames[:T])
        return out

    # ---- Sequence interface ------------------------------------------------

    def __len__(self):
        if len(self.df) == 0:
            return 0
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        X = np.zeros((len(batch), self.sequence_length, self.num_features),
                     dtype=np.float32)

        for i, (_, row) in enumerate(batch.iterrows()):
            name   = str(row.get(self.COL_NAME, ''))
            frames = load_sentence_frames(name, self.json_dir)
            X[i]   = self._pad_frames(frames)

        # Labels: return raw sentences for now (tokenizer can be wired later)
        sentences = batch.get(self.COL_SENTENCE,
                              pd.Series([''] * len(batch))).fillna('').tolist()
        return X, sentences


# ---------------------------------------------------------------------------
# Smoke-test: instantiate generators and peek at one batch
# ---------------------------------------------------------------------------
train_gen = How2SignGenerator(CSV_TRAIN, JSON_DIR_TRAIN, batch_size=4)
val_gen   = How2SignGenerator(CSV_VAL,   JSON_DIR_VAL,   batch_size=4)
test_gen  = How2SignGenerator(CSV_TEST,  JSON_DIR_TEST,  batch_size=4)

print(f'\nTrain batches : {len(train_gen)}')
print(f'Val   batches : {len(val_gen)}')
print(f'Test  batches : {len(test_gen)}')

if len(train_gen) > 0:
    X_sample, Y_sample = train_gen[0]
    print(f'\nSample batch X shape : {X_sample.shape}')
    print(f'Sample batch Y (first): {Y_sample[0]}')


⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv
⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv
⚠️  CSV not found: /kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv

Train batches : 0
Val   batches : 0
Test  batches : 0


## 4. Continuous Model Architecture (Seq2Seq)

In [9]:
# ============================================================
# 4. CONTINUOUS SEQ2SEQ ARCHITECTURE
# ============================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, BatchNormalization, SpatialDropout1D, Masking, TimeDistributed

vocab_size = 5000 

model_continuous = Sequential([
    Input(shape=(None, 78)), 
    Masking(mask_value=0.0),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    TimeDistributed(Dense(vocab_size, activation='softmax'))
])

model_continuous.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_1 (Masking)             │ (None, None, 78)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, None, 78)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 256)      │       211,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, None, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, None, 256)      │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, None, 5000)     │     1,285,000 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,892,232 (7.22 MB)

 Trainable params: 1,891,720 (7.22 MB)

 Non-trainable params: 512 (2.00 KB)

## 5. Tokenizer — Building the Vocabulary
We use Keras `TextVectorization` to convert raw sentences into token sequences. 
For CTC loss, we reserve index `0` for the `<blank>` token.


In [10]:
# ============================================================
# 5. TOKENIZER
# ============================================================
from tensorflow.keras.layers import TextVectorization
import pickle

# Collect all training sentences
train_df = pd.read_csv(CSV_TRAIN, sep='\t', on_bad_lines='skip')
if 'SENTENCE' not in train_df.columns:
    train_df = pd.read_csv(CSV_TRAIN, on_bad_lines='skip')

train_sentences = train_df['SENTENCE'].fillna('').tolist()

MAX_TOKENS = 5000
BLANK_INDEX = 0

# Create TextVectorization layer
# We leave index 0 empty for CTC blank
tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)

# Adapt on training sentences
print("Adapting tokenizer...")
tokenizer.adapt(train_sentences)

vocab = tokenizer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")
print(f"Top 10 words: {vocab[:10]}")

# Helper functions for encode/decode
# Shift indices by 1 to reserve 0 for BLANK
def encode_sentence(text):
    indices = tokenizer([text])[0].numpy()
    return indices + 1

def decode_indices(indices):
    words = []
    for idx in indices:
        if idx == 0:
            continue # blank
        idx = idx - 1
        if 0 <= idx < len(vocab):
            w = vocab[idx]
            if w not in ['', '[UNK]']:
                words.append(w)
    return ' '.join(words)

# Smoke test
sample_text = train_sentences[0] if len(train_sentences) > 0 else "hello world"
encoded = encode_sentence(sample_text)
decoded = decode_indices(encoded)
print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)



FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv'

## 6. Enhanced Feature Engineering (131-dim)
Instead of raw coordinates, we compute wrist-relative normalized coordinates and joint angles.
This makes the features translation-invariant.
- Body: 25 * 2 = 50
- Hands: 21 * 2 * 2 = 42
- Angles: 14
- Confidences: 25
Total: 131 dimensions.


In [ ]:
# ============================================================
# 6. FEATURE ENGINEERING (131-DIM WITH 14 ANGLES)
# ============================================================
import math
import numpy as np

def get_angle(p1, p2, p3):
    if p1[2] == 0 or p2[2] == 0 or p3[2] == 0: return 0.0
    dx1, dy1 = p1[0] - p2[0], p1[1] - p2[1]
    dx2, dy2 = p3[0] - p2[0], p3[1] - p2[1]
    angle1 = math.atan2(dy1, dx1)
    angle2 = math.atan2(dy2, dx2)
    return angle1 - angle2

def convert_openpose_to_131dim(pose, left_hand, right_hand):
    features = []
    # 1. Body (25 joints)
    neck = pose[1]
    torso_length = np.linalg.norm(pose[1][:2] - pose[8][:2]) if pose[8][2] > 0 else 1.0
    for i in range(25):
        if pose[i][2] > 0 and neck[2] > 0 and torso_length > 0:
            features.extend([(pose[i][0] - neck[0]) / torso_length, (pose[i][1] - neck[1]) / torso_length])
        else:
            features.extend([0.0, 0.0])
            
    # 2. Hands
    def process_hand(hand):
        wrist = hand[0]
        palm_width = np.linalg.norm(hand[0][:2] - hand[9][:2]) if hand[9][2] > 0 else 1.0
        for i in range(21):
            if hand[i][2] > 0 and wrist[2] > 0 and palm_width > 0:
                features.extend([(hand[i][0] - wrist[0]) / palm_width, (hand[i][1] - wrist[1]) / palm_width])
            else:
                features.extend([0.0, 0.0])
    process_hand(left_hand)
    process_hand(right_hand)
    
    # 3. 14 Joint Angles
    angles = []
    angles.append(get_angle(pose[1], pose[2], pose[3])) # RShoulder
    angles.append(get_angle(pose[1], pose[5], pose[6])) # LShoulder
    angles.append(get_angle(pose[2], pose[3], pose[4])) # RElbow
    angles.append(get_angle(pose[5], pose[6], pose[7])) # LElbow
    angles.append(get_angle(pose[3], pose[4], right_hand[0])) # RWrist
    angles.append(get_angle(pose[6], pose[7], left_hand[0]))  # LWrist
    angles.append(get_angle(pose[8], pose[9], pose[10])) # RHip
    angles.append(get_angle(pose[8], pose[12], pose[13])) # LHip
    angles.append(get_angle(pose[9], pose[10], pose[11])) # RKnee
    angles.append(get_angle(pose[12], pose[13], pose[14])) # LKnee
    angles.append(get_angle(pose[0], pose[1], pose[8])) # Neck
    angles.append(get_angle(pose[2], pose[1], pose[5])) # Shoulders Yaw
    angles.append(get_angle(pose[9], pose[8], pose[12])) # Hips Yaw
    angles.append(get_angle(pose[1], pose[8], pose[10])) # Torso-Leg
    features.extend(angles)
    
    # 4. Confidences
    for i in range(25): features.append(pose[i][2])
    return np.array(features)



## 6.5 Data Packer (Fast I/O)
To avoid the 11s/step disk bottleneck, we pack all `.npy` files into a single HDF5 dataset. Run this once locally before uploading.


In [ ]:
# ============================================================
# 6.5 HDF5 DATA PACKER
# ============================================================
import os
import h5py
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

h5_path = 'train_features.h5'
csv_path = 'how2sign_realigned_train.csv'
base_path = 'train_features/'

if not os.path.exists(h5_path) and os.path.exists(csv_path):
    print("Packing .npy files into HDF5 for fast I/O...")
    df = pd.read_csv(csv_path)
    with h5py.File(h5_path, 'w') as hf:
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            name = str(row.get('SENTENCE_NAME', ''))
            npy_file = os.path.join(base_path, f"{name}.npy")
            if os.path.exists(npy_file):
                data = np.load(npy_file)
                hf.create_dataset(name, data=data, compression="lzf")
    print("Packing complete! Generator will now use HDF5.")
else:
    print("HDF5 file already exists or CSV not found. Ready.")



## 7. CTC-Ready Data Generator
The data generator must yield inputs and label sequences, along with their lengths, to be used by CTC loss.


In [ ]:
# ============================================================
# 7. CTC DATA GENERATOR
# ============================================================
import h5py
import os
import numpy as np
import tensorflow as tf

class How2SignCTCGenerator(tf.keras.utils.Sequence):
    def __init__(self, csv_path, base_path, batch_size=16, sequence_length=300, max_label_len=50, augment=False):
        import pandas as pd
        if not str(csv_path).endswith('.csv') and not os.path.exists(csv_path):
            self.df = pd.DataFrame()
        else:
            try:
                self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
                if 'SENTENCE_NAME' not in self.df.columns:
                    self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            except:
                self.df = pd.DataFrame()
        self.base_path = base_path
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.max_label_len = max_label_len
        self.augment = augment
        
        self.h5_path = 'train_features.h5'
        self.use_h5 = os.path.exists(self.h5_path)
        if self.use_h5:
            self.h5_file = h5py.File(self.h5_path, 'r')
            
    def load_frames(self, video_name):
        if self.use_h5 and video_name in self.h5_file:
            return self.h5_file[video_name][:]
        else:
            p = os.path.join(self.base_path, f"{video_name}.npy")
            return np.load(p) if os.path.exists(p) else np.zeros((1, 131))

    def __len__(self):
        return int(np.ceil(len(self.df) / float(self.batch_size)))
        
    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        X = np.zeros((len(batch), self.sequence_length, 131), dtype=np.float32)
        Y = np.zeros((len(batch), self.max_label_len), dtype=np.int32)
        input_lengths = np.zeros((len(batch), 1), dtype=np.int32)
        label_lengths = np.zeros((len(batch), 1), dtype=np.int32)
        
        for i, (_, row) in enumerate(batch.iterrows()):
            name = str(row.get('SENTENCE_NAME', ''))
            frames = self.load_frames(name)
            
            T = min(len(frames), self.sequence_length)
            if T == 0: T = 1
            else:
                data = frames[:T]
                # Augmentation
                if self.augment:
                    data = data + np.random.normal(0, 0.005, data.shape) # Spatial Jitter
                    data = data * np.random.uniform(0.95, 1.05) # Scale shift
                X[i, :T, :] = data
                
            sentence = str(row.get('SENTENCE', ''))
            encoded = encode_sentence(sentence)
            L = min(len(encoded), self.max_label_len)
            
            if L > T: L = T
            if L > 0: Y[i, :L] = encoded[:L]
            
            input_lengths[i, 0] = T
            label_lengths[i, 0] = L
            
        return tuple([X, Y, input_lengths, label_lengths]), np.zeros((len(batch),), dtype=np.float32)

train_gen_ctc = How2SignCTCGenerator(CSV_TRAIN, base_path, batch_size=16, sequence_length=300, augment=True)
val_gen_ctc = How2SignCTCGenerator(CSV_VAL, base_path, batch_size=16, sequence_length=300, augment=False)



## 8. CTC BiLSTM Architecture
We use a custom CTC layer to compute the loss during training.


In [ ]:
# ============================================================
# 8. CTC MODEL ARCHITECTURE WITH ATTENTION
# ============================================================
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Bidirectional, Layer, Input, MultiHeadAttention, LayerNormalization
from tensorflow.keras.models import Model
import tensorflow as tf

class CTCLossLayer(Layer):
    def __init__(self, name=None):
        super().__init__(name=name)
    def call(self, y_true, y_pred, input_length, label_length):
        loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
        self.add_loss(tf.reduce_mean(loss))
        return loss

input_layer = Input(shape=(300, 131), name='input')
labels = Input(shape=(50,), name='labels')
input_length = Input(shape=(1,), name='input_length')
label_length = Input(shape=(1,), name='label_length')

x1 = Bidirectional(LSTM(256, return_sequences=True))(input_layer)
x1 = BatchNormalization()(x1)

# Multi-Head Attention Mechanism
attn = MultiHeadAttention(num_heads=4, key_dim=64)(x1, x1)
x = LayerNormalization()(x1 + attn) # Residual Add
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(256, return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

vocab_size_ctc = len(vocab) + 1
output = Dense(vocab_size_ctc, activation='softmax', name='prediction')(x)

loss_out = CTCLossLayer(name='ctc_loss')(labels, output, input_length, label_length)

model_ctc_train = Model(inputs=[input_layer, labels, input_length, label_length], outputs=loss_out)
model_ctc_train.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0))

model_inference = Model(inputs=input_layer, outputs=output)
model_ctc_train.summary()



## 9. Training
Training with EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint for 30 epochs.


In [ ]:
# ============================================================
# 9. TRAINING LOOP
# ============================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ModelCheckpoint('best_cslr_model.weights.h5', save_best_only=True, save_weights_only=True)
]

history = model_ctc_train.fit(
     train_gen_ctc,
     validation_data=val_gen_ctc,
     epochs=30,
     callbacks=callbacks
 )

print("Model training ready (epochs=30).")



## 10. Evaluation & Metrics
Calculating Word Error Rate (WER), Precision, Recall, F1, and PR-AUC.


In [ ]:
pip install jiwer


In [ ]:
# ============================================================
# 10. EVALUATION & METRICS
# ============================================================
# !pip install jiwer
import jiwer
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import tensorflow.keras.backend as K

def ctc_greedy_decode(logits, input_lengths):
    decoded, _ = tf.keras.backend.ctc_decode(logits, input_length=input_lengths, greedy=True)
    return decoded[0].numpy()

def evaluate_model(model_inf, test_gen):
    print("Evaluating model...")
    all_hypotheses = []
    all_references = []
    
    max_batches = min(10, len(test_gen))
    for i in range(max_batches):
        batch_x, _ = test_gen[i]
        logits = model_inf.predict(batch_x['input'], verbose=0)
        decoded = ctc_greedy_decode(logits, batch_x['input_length'].flatten())
        
        for j in range(len(decoded)):
            hyp_indices = [idx for idx in decoded[j] if idx != -1]
            hyp_str = decode_indices(hyp_indices)
            all_hypotheses.append(hyp_str)
            
            ref_indices = [idx for idx in batch_x['labels'][j] if idx != 0]
            ref_str = decode_indices(ref_indices)
            all_references.append(ref_str)
            
    all_hypotheses = [h if len(h) > 0 else "empty" for h in all_hypotheses]
    all_references = [r if len(r) > 0 else "empty" for r in all_references]
    
    wer = jiwer.wer(all_references, all_hypotheses)
    print(f"Word Error Rate (WER): {wer:.4f}")
    
    return all_references, all_hypotheses

test_gen_ctc = How2SignCTCGenerator(CSV_TEST, 'train_features/', batch_size=16, sequence_length=300, augment=False)
# refs, hyps = evaluate_model(model_inference, test_gen_ctc)




## 11. Stabilization Tracker
A sliding window approach with majority voting to smooth out raw CTC predictions in real-time.


In [ ]:
# ============================================================
# 11. STABILIZATION TRACKER
# ============================================================
from collections import deque
import time

class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s = cooldown_s
        self.buffer = deque(maxlen=window_size)
        self.last_commit_time = 0
        self.last_committed_word = ""
        self.sentence = []

    def update(self, predicted_word):
        if not predicted_word:
            self.buffer.append(None)
            return None
            
        self.buffer.append(predicted_word)
        
        if len(self.buffer) < self.window_size:
            return None
            
        counts = {}
        for w in self.buffer:
            if w: counts[w] = counts.get(w, 0) + 1
            
        if not counts: return None
        
        top_word = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time = now
                self.buffer.clear()
                return top_word
        return None

tracker = StabilizationTracker()



## 12. Real-Time MediaPipe Webcam Inference
Webcam demo utilizing MediaPipe Holistic. We extract landmarks, match OpenPose indices, buffer frames, predict with the trained sequence model, and stabilize the output.
Both modes (live webcam and offline video file) are supported. Minimal latency is ensured by sliding the buffer window.


In [ ]:
pip install mediapipe==0.10.14


In [ ]:
# ============================================================
# 12. WEBCAM INFERENCE LOOP
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from collections import deque

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_to_131dim(results):
    # Map MP Holistic to OP Body_25
    pose = np.zeros((25, 3))
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        
        def set_p(op_idx, mp_idx):
            if mp_idx < len(lm):
                pose[op_idx] = [lm[mp_idx].x, lm[mp_idx].y, lm[mp_idx].visibility]
                
        set_p(0, 0)   # Nose
        set_p(2, 12)  # RShoulder
        set_p(3, 14)  # RElbow
        set_p(4, 16)  # RWrist
        set_p(5, 11)  # LShoulder
        set_p(6, 13)  # LElbow
        set_p(7, 15)  # LWrist
        set_p(9, 24)  # RHip
        set_p(10, 26) # RKnee
        set_p(11, 28) # RAnkle
        set_p(12, 23) # LHip
        set_p(13, 25) # LKnee
        set_p(14, 27) # LAnkle
        set_p(15, 5)  # REye
        set_p(16, 2)  # LEye
        set_p(17, 8)  # REar
        set_p(18, 7)  # LEar
        set_p(19, 31) # LBigToe
        set_p(21, 29) # LHeel
        set_p(22, 32) # RBigToe
        set_p(24, 30) # RHeel
        
        # Interpolate Neck (1) between shoulders
        if pose[2][2] > 0 and pose[5][2] > 0:
            pose[1] = (pose[2] + pose[5]) / 2.0
            
        # Interpolate MidHip (8) between hips
        if pose[9][2] > 0 and pose[12][2] > 0:
            pose[8] = (pose[9] + pose[12]) / 2.0

    left_hand = np.zeros((21, 3))
    if results.left_hand_landmarks:
        for i, pt in enumerate(results.left_hand_landmarks.landmark):
            left_hand[i] = [pt.x, pt.y, 1.0]

    right_hand = np.zeros((21, 3))
    if results.right_hand_landmarks:
        for i, pt in enumerate(results.right_hand_landmarks.landmark):
            right_hand[i] = [pt.x, pt.y, 1.0]

    return convert_openpose_to_131dim(pose, left_hand, right_hand)

def run_inference(source=0):
    cap = cv2.VideoCapture(source)
    tracker = StabilizationTracker(window_size=15, majority_ratio=0.6)
    
    # Updated sequence buffer to 300 to match new model pad
    frame_buffer = deque(maxlen=300)
    
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            
            features = mediapipe_to_131dim(results)
            frame_buffer.append(features)
            
            if len(frame_buffer) > 10:
                X = np.expand_dims(np.stack(frame_buffer), axis=0)
                if X.shape[1] < 300:
                    pad = np.zeros((1, 300 - X.shape[1], 131))
                    X = np.concatenate([X, pad], axis=1)
                
                preds = model_inference.predict(X, verbose=0)
                input_lengths = np.array([min(len(frame_buffer), 300)])
                decoded, _ = tf.keras.backend.ctc_decode(preds, input_length=input_lengths, greedy=True)
                indices = decoded[0][0].numpy()
                
                sentence = decode_indices(indices)
                tracker.update(sentence)
                
            final_text = tracker.get_stable_sentence()
            cv2.putText(image, final_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow('Sign Language Translation', image)
            
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break
                
    cap.release()
    cv2.destroyAllWindows()
    
# run_inference(0)



## 13. Export to ONNX
We convert the inference model to ONNX format for deployment.

In [ ]:
pip install tf2onnx


In [ ]:
# ============================================================
# 13. EXPORT TO ONNX
# ============================================================
# Uncomment the line below if tf2onnx is not installed
# !pip install tf2onnx
import tensorflow as tf
import tf2onnx
import onnx

# Load best weights into the inference model
try:
    model_inference.load_weights('best_cslr_model.weights.h5')
    print("Loaded best weights for ONNX export.")
except Exception as e:
    print("Could not load weights. Ensure the training cell has finished successfully.", e)

# Define input signature: batch size 1, 150 frames, 131 features
input_signature = [tf.TensorSpec([1, 150, 131], tf.float32, name='input')]

print("Converting model to ONNX...")
# Convert the inference model (which lacks the CTC loss layer) to ONNX
onnx_model, _ = tf2onnx.convert.from_keras(model_inference, input_signature, opset=13)

# Save
onnx_path = "cslr_inference_model.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"ONNX model successfully saved to {onnx_path}")

